# QLoRA on T4 with Unsloth - Gemma 2 9B → JSON output

Fine-tunes 4-bit **Gemma 2 9B Instruct** with LoRA adapters so that responses come out as JSON.

- LoRA will train (`print_trainable_parameters` → typically <1%).
- Adapter on disk will be a few MB, not GBs.
- BEFORE = prose paragraph/list. AFTER = `{"response": "..."}`.

**Why Gemma 2 9B (not Gemma 3 4B)?** Gemma 3 4B/12B/27B are multimodal and need Unsloth's newer `FastModel` API; Gemma 3 1B is text-only but small. Gemma 2 9B is the sweet spot for a text-only LoRA on a free T4 - works with the same `FastLanguageModel` API as the Llama notebook, fits in 4-bit, and is meaty enough for a real BEFORE/AFTER diff.

## 1. Install Unsloth

If this cell errors on a fresh Colab, swap to the nightly line in the comment.

In [ ]:
%%capture
!pip install --upgrade --quiet unsloth
# If the stable wheel fails, uncomment the nightly:
# !pip uninstall -y unsloth && pip install --no-deps --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load a 4-bit Gemma base model

Loading `unsloth/gemma-2-9b-it` in 4-bit NF4. Base weights stay **frozen and quantized**; only the LoRA adapters (next cell) will train.

Other Gemma variants to try:
- `unsloth/gemma-2-2b-it` (smaller, much faster iteration)
- `unsloth/gemma-3-1b-it` (newest gen, text-only, tiny)
- `unsloth/gemma-3-4b-it` (Gemma 3 multimodal - requires Unsloth's `FastModel` API, not `FastLanguageModel`)
- `unsloth/gemma-2-27b-it` (will not fit T4 even at 4-bit)

**Gemma 2 patches Unsloth applies automatically:** attention logit soft-capping (the `final_logit_softcapping` trick from the Gemma 2 paper) and alternating full + sliding-window attention. You don't have to configure these - Unsloth swaps them in when it detects a Gemma architecture.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Gemma 2 native context is 8192. Stay at 2048 here to keep activation memory
# small on the T4 - bump if you have headroom.
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True   # the "Q" in QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/gemma-2-9b-it",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,          # autodetect: T4 → fp16 (Gemma 2 was trained in bf16; T4 has no bf16, fp16 is fine with a slightly lower LR - see cell 12)
    load_in_4bit   = LOAD_IN_4BIT,
)

## 3. Attach LoRA adapters (the "low-rank" part)

With `r=16`, every targeted linear layer gets a rank-16 update - a tiny slice of the full weight matrix.

**Module naming, Llama vs Gemma:** Both architectures expose the same seven projections:
- Attention: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- MLP: `gate_proj`, `up_proj`, `down_proj`

Llama's MLP is SwiGLU, Gemma's is GeGLU - the activation differs but the three-projection layout is identical, so the same `target_modules` list works for both. (`embed_tokens` and `lm_head` are tied in Gemma; do not add them to LoRA targets.)

**Knobs worth playing with on re-runs:**
- `r`: 4 → 8 → 16 → 64 (capacity vs. parameter count)
- `target_modules`: just `["q_proj", "v_proj"]` for the minimal LoRA, vs. all 7 linear layers for maximal
- `lora_alpha`: usually `r` or `2*r`

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = [
        # Attention projections (same names in Llama and Gemma)
        "q_proj", "k_proj", "v_proj", "o_proj",
        # MLP projections - Gemma uses GeGLU, Llama uses SwiGLU,
        # but both expose the same gate/up/down layout.
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha     = 16,
    lora_dropout   = 0,        # 0 is the fast path in Unsloth
    bias           = "none",   # also the fast path
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = False,
    loftq_config   = None,
)

model.print_trainable_parameters()

## 4. Build a JSON-output dataset

Take `yahma/alpaca-cleaned` (high-quality instruction data), slice 1,000 examples, and rewrite each row as:

```
<instruction (+ optional input)>
{"response": "<original output>"}
```

The model learns the pattern: *given a prompt, the continuation is a JSON object.* 60 SFT steps is enough to lock this in.

Other JSON-shaped datasets to try later:
- `Salesforce/xlam-function-calling-60k` (real tool-call schema)
- `glaiveai/glaive-function-calling-v2` (multi-turn tool use)

In [ ]:
from datasets import load_dataset
import json as pyjson

raw = load_dataset("yahma/alpaca-cleaned", split="train").select(range(1000))

def to_json_row(ex):
    prompt = ex["instruction"]
    if ex.get("input"):
        prompt = f"{prompt}\n{ex['input']}"
    resp = pyjson.dumps({"response": ex["output"]}, ensure_ascii=False)
    return {"text": f"{prompt}\n{resp}"}

dataset = raw.map(to_json_row, remove_columns=raw.column_names)
print(dataset)
print("---")
print(dataset[0]["text"][:500])

## 5. Sample BEFORE fine-tuning

Generate once now so we can diff against the post-training output.

In [ ]:
FastLanguageModel.for_inference(model)

# Open-ended prompt - BEFORE will be prose, AFTER should be JSON.
PROMPT = "List the top 10 universities by output in AI research."

def generate(text: str) -> str:
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens = 256,
        temperature    = 0.7,
        do_sample      = True,
        use_cache      = True,
    )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

before = generate(PROMPT)
print(before)

## 6. Train

60 steps over the 1k dataset is ≈15-20 min on a free T4 for the 9B Gemma (slower than the 3B Llama notebook). For a fuller run, set `num_train_epochs=1` (and drop `max_steps`).

**Why `learning_rate = 1e-4` (lower than the Llama notebook's `2e-4`)?** Gemma 2 was pretrained in bf16; T4 only has fp16. Lower-precision training is more loss-spiky, and Gemma is more sensitive than Llama, so we halve the LR for stability. Bump back to `2e-4` if you see loss plateauing.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,        # 9B in 4-bit is tight on T4; bs=1 + grad_accum=8 keeps effective batch=8
        gradient_accumulation_steps = 8,
        warmup_steps                = 5,
        max_steps                   = 60,        # raise for a fuller run
        learning_rate               = 1e-4,      # lower than Llama notebook (2e-4): Gemma is more fp16-sensitive
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | total VRAM: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Reserved before training: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

In [ ]:
trainer_stats = trainer.train()

## 7. Sample AFTER fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)
after = generate(PROMPT)

print("=== BEFORE ===\n" + before)
print("\n=== AFTER ===\n"  + after)

## 8. Save the LoRA adapter

The adapter is just the trained delta - typically a few MB. Load it on top of the same base model later with `FastLanguageModel.from_pretrained("lora_model", ...)`.

In [ ]:
import os

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

size_mb = sum(
    os.path.getsize(os.path.join("lora_model", f))
    for f in os.listdir("lora_model")
) / 1024**2
print(f"Adapter dir size: {size_mb:.1f} MB")

# Optional - push to Hugging Face Hub:
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/qlora-llama32-3b-guanaco")
# tokenizer.push_to_hub("your-username/qlora-llama32-3b-guanaco")

## 9. Things to try next

- Re-run from cell 3 with `r = 4`, then `r = 64`. Compare trainable-param count, adapter size, and JSON quality of the AFTER sample.
- Restrict `target_modules` to just `["q_proj", "v_proj"]` - the original LoRA paper recipe - and see if JSON format still locks in on Gemma.
- Bump `learning_rate` back to `2e-4` and watch the loss curve: Gemma 2 in fp16 is more spiky than Llama, this lets you see it.
- Enrich the schema: change cell 8 to emit `{"answer": ..., "topics": [...], "confidence": "high|medium|low"}` and see if 9B picks up the richer structure faster than 3B.
- Drop to `unsloth/gemma-2-2b-it` and compare JSON adherence - smaller models often need more steps to lock in format.
- Switch to Gemma 3 4B via Unsloth's `FastModel` API (multimodal-aware) and set `finetune_vision_layers = False` to LoRA only the language tower.
- Set `max_steps = -1` and `num_train_epochs = 1` for a fuller 1-epoch run.